# Notebook 02 — Classification Results
**Purpose:** Evaluate propensity classifiers and translate scores into business value  

---
## What this notebook answers
1. Which products have reliable propensity models (AUC, Lift@10%)?
2. What features drive each model?
3. For a given marketing budget, how many more buyers do we find vs. random?
4. Where does the model make the most mistakes (confusion analysis)?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
pd.set_option('display.float_format', '{:.4f}'.format)

from src.config import get_config
from src.features.preprocessing import run_preprocessing

cfg = get_config()
print('Loading features...')
X, df_hh = run_preprocessing()
print(f'Shape: {X.shape}  |  Households: {len(df_hh)}')

## 1. Train All Product Classifiers

In [ ]:
from src.models.classifier import train_all_products

summary = train_all_products(X, df_hh, cfg=cfg)
print(summary[['product','model','roc_auc','f1','precision','recall','lift_at_10']].to_string(index=False))

## 2. Model Comparison Dashboard

In [ ]:
if not summary.empty:
    # Best model per product
    best = summary.loc[summary.groupby('product')['roc_auc'].idxmax()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # AUC comparison
    colors_auc = ['#2E4A8C' if v >= 0.70 else '#E84C3D' for v in best['roc_auc']]
    axes[0].barh(best['product'], best['roc_auc'],
                 color=colors_auc, edgecolor='white')
    axes[0].axvline(x=0.70, color='#F39C12', linestyle='--', lw=1.8, label='0.70 target')
    axes[0].set_xlabel('ROC-AUC', fontsize=11)
    axes[0].set_title('Best AUC per Product', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=9)
    axes[0].set_xlim(0.5, 1.0)

    # Lift@10% comparison
    colors_lift = ['#2E4A8C' if v >= 2.0 else '#E84C3D' for v in best['lift_at_10']]
    axes[1].barh(best['product'], best['lift_at_10'],
                 color=colors_lift, edgecolor='white')
    axes[1].axvline(x=2.0, color='#F39C12', linestyle='--', lw=1.8, label='2.0× target')
    axes[1].set_xlabel('Lift @ Top 10%', fontsize=11)
    axes[1].set_title('Lift@10% per Product', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=9)

    for ax in axes:
        ax.spines[['top','right']].set_visible(False)

    plt.suptitle('Classification Model Performance Summary',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(cfg.paths.report_dir + 'model_comparison.png', bbox_inches='tight')
    plt.show()

## 3. Business Value Translation
How many additional buyers does the model find vs. random outreach?

In [ ]:
if not summary.empty:
    best = summary.loc[summary.groupby('product')['roc_auc'].idxmax()].copy()

    total_hh      = len(df_hh)
    contact_pct   = 0.10  # top 10% campaign
    n_contact     = int(total_hh * contact_pct)
    cost_per_contact = 10  # $10 per outreach contact
    campaign_budget  = n_contact * cost_per_contact

    print(f'Campaign assumptions:')
    print(f'  Total households:     {total_hh:,}')
    print(f'  Contacts (top 10%):   {n_contact:,}')
    print(f'  Cost per contact:     ${cost_per_contact}')
    print(f'  Total budget:         ${campaign_budget:,}')
    print()
    print(f'{"Product":<28} {"Base Rate":>10} {"Random Finds":>13} {"Model Finds":>12} {"Uplift":>8}')
    print('-' * 75)

    for _, row in best.iterrows():
        prod     = row['product']
        if prod not in df_hh.columns:
            continue
        base_rate   = df_hh[prod].mean()
        random_find = int(n_contact * base_rate)
        model_find  = int(n_contact * base_rate * row['lift_at_10'])
        uplift      = model_find - random_find
        print(f'  {prod:<26} {base_rate:>9.1%} {random_find:>13,} {model_find:>12,} +{uplift:>6,}')

## 4. Saved Model Inventory

In [ ]:
import os
import json

model_dir = cfg.paths.model_dir
meta_files = sorted([f for f in os.listdir(model_dir) if f.endswith('_metadata.json')])

print(f'Saved models in {model_dir}:')
for mf in meta_files:
    with open(os.path.join(model_dir, mf)) as f:
        meta = json.load(f)
    label   = meta.get('label', mf)
    auc_val = meta.get('roc_auc', meta.get('r2', 'N/A'))
    n_feat  = meta.get('feature_count', 'N/A')
    print(f'  {label:<40}  metric={auc_val}  features={n_feat}')